# 01.02 · Limpieza y troceado incremental

Crea chunks deterministas únicamente para transcripciones nuevas o modificadas y conserva la versión del troceador.

La normalización NFKC aplicada al texto sigue las formas de normalización Unicode [1], y las huellas de transcripción, texto e identificadores estables usan SHA-256 [2]. Las ventanas de 30 s, los límites de caracteres, el solapamiento y las reglas de deduplicación son parámetros locales versionados: las fuentes anteriores no demuestran que esos valores sean óptimos.

**Contrato v2.1:** `SEGURO` + cuatro daños entrenados, incluida `ATAQUE_POR_GENERO_IDENTIDAD`. `SEGURO` es excluyente; los daños son multietiqueta. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('Proyecto:', ROOT)


## Configuración

In [ ]:
from moderacion_peru.incremental import chunk_records_incrementally
from moderacion_peru.io import append_jsonl_once, read_jsonl
SOURCE=ROOT/'datos/raw/transcripts_raw.jsonl'
OUTPUT=ROOT/'datos/processed/chunks_v2.jsonl'
VERSION_INDEX=ROOT/'datos/processed/chunking_v2_versions.jsonl'

## Materialización

In [ ]:
existing=list(read_jsonl(OUTPUT)) if OUTPUT.exists() else []
versions=list(read_jsonl(VERSION_INDEX)) if VERSION_INDEX.exists() else []
new_rows,new_versions,stats=chunk_records_incrementally(read_jsonl(SOURCE) if SOURCE.exists() else [],existing,versions)
added,skipped=append_jsonl_once(OUTPUT,new_rows,id_field='chunk_id')
versions_added,_=append_jsonl_once(VERSION_INDEX,new_versions,id_field='version_id')
stats.update({'added':added,'duplicate_ids':skipped,'versions_registered':versions_added})
print(stats)

## Referencias

[1] Unicode Consortium, "Unicode Normalization Forms," Unicode Standard Annex No. 15, rev. 57, Jul. 2025. [Online]. Available: https://www.unicode.org/reports/tr15/tr15-57.html

[2] National Institute of Standards and Technology, "Secure Hash Standard (SHS)," FIPS PUB 180-4, Aug. 2015, doi: 10.6028/NIST.FIPS.180-4.